In [ ]:
import numpy as np
import random

from matplotlib import pyplot as plt
import cv2

import time
import pandas as pd

import gc

#root = '/home/a01328525/'
root = 'E:/Experiments/'
root = '../'

folder_save_results = root+'Counting images paper 2/'

## Statistical Evaluation

In [ ]:
## Zone A ---> Zone1
## Zone B ---> Zone3
## Zone C ---> Zone109
## Zone D ---> Zone108
## Zone E ---> Zone108_octubre
## Zone F ---> Zone102_part1

zone_name = "Zone108_octubre"
feats_all = pd.read_csv(folder_save_results+zone_name+'_agave_features_final.csv',index_col=0)

In [ ]:
len(feats_all['Target']),len(feats_all['Target'][feats_all['Target']==0])

In [ ]:
# Convert to integer pixel indices
x = feats_all['xc'].to_numpy().astype(int)
y = feats_all['yc'].to_numpy().astype(int)

In [ ]:
feats_all["Target2"] = np.where(feats_all.ARVI > 0.5, 1, 0)
feats_all["Target2"]


In [ ]:
import seaborn as sb

df = feats_all.copy()
cols = ['A', 'P', 'Compactness', 'Circularity', 'Effect_diameter',
       'Shape_index', 'Contrast', 'Dissimilarity', 'Homogeneity', 'ASM',
       'Energy', 'NDVI', 'PVI1', 'PVI2', 'SAVI', 'TSAVI2', 'NDGI', 'ARVI',
       'NDCI', 'MSAVI', 'NLI', 'RDVI', 'GNDVI', 'OSAVI', 'MSR', 'EVI', 'EVI2']
cols = ['A', 'P', 'Compactness', 'Circularity', 'Solidity', 
                  'Contrast', 'Dissimilarity', 'Homogeneity', 'ASM', 'Energy', 
                  'ARVI', 'NDCI', 'TSAVI2', 'NDVI', 'GNDVI']

anomalies = df[cols][df['Target']==0].values
normal = df[cols][df['Target']!=0].values
print("Anomalies: %4d, Normal: %4d"%(len(anomalies), len(normal)))

#filtered_df = pd.DataFrame(np.vstack([anomalies, normal]), columns = cols)
#filtered_df["Target"] = np.hstack(([[0]*len(anomalies), [1]*len(normal)]))

#df = filtered_df
########### RANDOM TARGET ###############
#random_t = np.random.choice([0, 1], size=len(df), p=[0.05, 0.95])
#df["Target"] = random_t
########### ARVI TARGET ###############
#df["Target"] = (df['ARVI'] < 0.5).astype('uint8')
#df["Target"] = (df['A'] < 70).astype('uint8')
#df["Target"] = (df['NDVI'] < 0.5).astype('uint8')
df

In [ ]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
X_scaled = scaler.fit_transform(df[cols])

df_scaled = df.copy()
df_scaled[cols] = X_scaled
df_scaled

###  Feature-level distributional separation

In [ ]:
from scipy.stats import ks_2samp, mannwhitneyu
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)
from sklearn.metrics import pairwise_distances

In [ ]:
# ===== Cluster Quality =====
def cluster_validity(df, feature_cols):
    X = df[feature_cols].values
    labels = df['Target'].values
    return {
        'silhouette': silhouette_score(X, labels, metric='mahalanobis'),
        'calinski_harabasz': calinski_harabasz_score(X, labels),
        'davies_bouldin': davies_bouldin_score(X, labels)
    }

In [ ]:
# ===== Distributional Tests =====
def distribution_tests(df, feature):
    x = df.loc[df['Target'] == 0, feature].values
    y = df.loc[df['Target'] == 1, feature].values
    ks_stat, ks_p = ks_2samp(x, y)
    u_stat, u_p = mannwhitneyu(x, y, alternative='two-sided')
    return {'ks_p': ks_p, 'mw_p': u_p}
    
# ===== Effect Size =====
def cliffs_delta(x, y):
    nx, ny = len(x), len(y)
    return (np.sum(x[:,None] > y) - np.sum(x[:,None] < y)) / (nx*ny)

def effect_sizes(df, feature):
    x = df.loc[df.Target == 0, feature].values
    y = df.loc[df.Target == 1, feature].values
    return {'cliffs_delta': cliffs_delta(x, y)}

In [ ]:
rows = []
# ---- Feature-wise statistical tests ----
for feat in cols:
    dist = distribution_tests(df, feat)
    eff = effect_sizes(df, feat)

    rows.append({
        "Feature": feat,
        "KS p-value": dist["ks_p"],
        "Mann–Whitney p-value": dist["mw_p"],
        "Cliff's Delta": eff["cliffs_delta"]
    })

feature_table = pd.DataFrame(rows)
#feature_table.to_csv(folder_save_results+zone_name+'_statistics_features.csv')
feature_table

In [ ]:
from statsmodels.stats.multitest import multipletests

# Example: 15 raw p-values from the KS tests in Zone E
reject_ks_e, p_holm_ks_e, _, _ = multipletests(
    feature_table["KS p-value"],
    alpha=0.05,
    method="holm"
)

# Mann--Whitney tests in Zone E
reject_mw_e, p_holm_mw_e, _, _ = multipletests(
    feature_table["Mann–Whitney p-value"],
    alpha=0.05,
    method="holm"
)

In [ ]:
feature_table["Holm KS"] = reject_ks_e
feature_table["Holm MW"] = reject_mw_e
feature_table

### Multivariate distributional divergence

In [ ]:
# ===== Multivariate Permutation Test =====
def permanova(X, labels, n_perm=1000):
    # centroids
    unique = np.unique(labels)
    group_means = [X[labels == u].mean(axis=0) for u in unique]
    ss_between = sum([
        len(X[labels == u]) * np.sum((m - X.mean(axis=0))**2)
        for u, m in zip(unique, group_means)
    ])
    stats = []
    pooled = X.copy()
    
    for _ in range(n_perm):
        perm = np.random.permutation(labels)
        group_means_perm = [X[perm == u].mean(axis=0) for u in unique]
        ssb_perm = sum([
            len(X[perm == u]) * np.sum((m - X.mean(axis=0))**2)
            for u, m in zip(unique, group_means_perm)
        ])
        stats.append(ssb_perm)
    p = (1 + np.sum(np.array(stats) >= ss_between)) / (1 + n_perm)
    return ss_between, p

In [ ]:
from hyppo.ksample import Energy
from scipy.stats import energy_distance
# Install with: pip install pted
from pted import pted
import numpy as np

repetitions = 1000

stat, p_energy = Energy().test(
    df[cols].loc[df.Target == 1].values,#normal
    df[cols].loc[df.Target == 0].values,#anomaly
    reps=repetitions
)

X = df_scaled[cols].values
y = df_scaled["Target"].values
sample1 = X[y == 0]
sample2 = X[y == 1]

# Perform the energy distance permutation test
# H0: The two samples are from the same distribution
p_penergy = pted(sample1, sample2, permutations=repetitions)

X = df_scaled[cols].values
labels = df_scaled['Target'].values
_, p_permanova = permanova(X, labels)


multivariate_metrics = pd.DataFrame([
    {"Metric": "Energy p-value", "Value": p_energy},
    {"Metric": "Energy Permutation p-value", "Value": p_penergy},
    {"Metric": "PERMANOVA p-value", "Value": p_permanova},
])
multivariate_metrics

In [ ]:
#%pip install hyppo
#%pip install pted
#%pip install "numpy<=2.3"

### Cluster structure

In [ ]:
from hyppo.ksample import Energy
from scipy.stats import energy_distance

validity = cluster_validity(df_scaled, cols)
#j_scores = bootstrap_target_jaccard(df["Target"].values)

global_metrics = pd.DataFrame([
    {"Metric": "Silhouette", "Value": validity["silhouette"]},
    {"Metric": "Calinski–Harabasz", "Value": validity["calinski_harabasz"]},
    {"Metric": "Davies–Bouldin", "Value": validity["davies_bouldin"]},
    #{"Metric": "Mean Jaccard stability", "Value": j_scores.mean()},
])
global_metrics

### Spatial test

#### Moran

In [ ]:
import numpy as np
import pandas as pd
from libpysal.weights import KNN
from esda.moran import Moran

# df must contain spatial coordinates
# columns: ['x', 'y', 'Target']
coords = df[['xc', 'yc']].values
labels = df['Target'].values  # 1 = anomaly, 0 = normal

# Spatial weights: k-nearest neighbors
w = KNN.from_array(coords, k=8)
w.transform = 'R'  # row-standardization

# Moran's I
moran = Moran(labels, w, permutations=999)

print(f"Moran's I: {moran.I:.3f}")
print(f"p-value (permutation): {moran.p_sim:.5f}")


In [ ]:
multivariate_metrics = pd.DataFrame([
    {"Metric": "Energy p-value", "Value": p_energy},
    {"Metric": "Energy Permutation p-value", "Value": p_penergy},
    {"Metric": "PERMANOVA p-value", "Value": p_permanova},
    
    {"Metric": "Mooran's I p_value", "Value": moran.p_sim},
    
    {"Metric": "Mooran's I", "Value": moran.I},
    
    {"Metric": "Silhouette", "Value": validity["silhouette"]},
    {"Metric": "Calinski–Harabasz", "Value": validity["calinski_harabasz"]},
    {"Metric": "Davies–Bouldin", "Value": validity["davies_bouldin"]},
])
multivariate_metrics.to_csv(folder_save_results+zone_name+'_statistics_all_features.csv')
multivariate_metrics

In [ ]:
#%pip install libpysal
#%pip install esda

#### Ripley

In [ ]:
from pointpats import k
# anomaly points
points = df.loc[df.Target == 0, ['xc', 'yc']].values
r = np.linspace(0, 40, 100)
#print(r)
k_sup, k_estimate = k(points, r)

#_, K = ripley_k(points, width, height, 40, 100)
#print(np.array(k_estimate-K).sum())
#print(K-k_estimate)
#print(K)

In [ ]:
import numpy as np
from scipy.spatial import KDTree

def ripley_k(points, width, height, r_max=500, n_steps=50):
    
    points = np.asarray(points)
    n = len(points)
    
    area = width * height
    density = n / area
    
    radii = np.linspace(0, r_max, n_steps)
    
    tree = KDTree(points)
    
    K = np.zeros(len(radii))
    
    for i, r in enumerate(radii):
        
        neighbors = tree.query_ball_tree(tree, r)
        
        count = sum(len(x) - 1 for x in neighbors)
        
        K[i] = count / (n * density)
    
    return radii, K


def ripley_l(K):
    return np.sqrt(K / np.pi)


def ripley_h(r, L):
    return L - r


def ripley_analysis(points, width, height,
                    r_max=500,
                    n_steps=50,
                    n_sim=100):

    #r, K = ripley_k(points, width, height, r_max, n_steps)
    r = np.linspace(0, r_max, n_steps)
    r, K = k(points, r)

    L = ripley_l(K)
    H = ripley_h(r, L)

    sims = []

    for _ in range(n_sim):

        rand_points = np.column_stack([
            np.random.uniform(0, width, len(points)),
            np.random.uniform(0, height, len(points))
        ])

        _, K_sim = ripley_k(rand_points, width, height, r_max, n_steps)
        L_sim = ripley_l(K_sim)
        H_sim = ripley_h(r, L_sim)

        sims.append(H_sim)

    sims = np.array(sims)

    lower = np.percentile(sims, 2.5, axis=0)
    upper = np.percentile(sims, 97.5, axis=0)

    auc_h = np.trapz(H, r)

    return {
        "r": r,
        "K": K,
        "L": L,
        "H": H,
        "lower_env": lower,
        "upper_env": upper,
        "AUC_H": auc_h
    }

In [ ]:
import matplotlib.pyplot as plt

def plot_ripley(result):

    r = result["r"]
    H = result["H"]

    plt.figure(figsize=(6,5))

    plt.plot(r, H, color = "red", lw=2, label="Observed H(r)")
    plt.fill_between(r,
                     result["lower_env"],
                     result["upper_env"],
                     alpha=0.3,
                     label="95% CSR envelope")

    plt.axhline(0, linestyle="--")

    plt.xlabel("Radius (pixels)")
    plt.ylabel("H(r) = L(r) - r")
    plt.legend()

    plt.title("Ripley Spatial Clustering")
    #plt.savefig(folder_save_results+zone_name+"_Ripley", dpi=300)
    
    plt.show()

In [ ]:
points_ano = coords[labels==0]

In [ ]:
result = ripley_analysis(
            points_ano,
            width=10000,#img_rgb.shape[0],
            height=10000,#img_rgb.shape[1],
            r_max=1000,
            n_steps=60,
            n_sim=200)

print("AUC_H:", result["AUC_H"])

plot_ripley(result)